# Estado de Dados Brasil: Engenharia de SPECS com PySpark & Dashboards Executivos
**Tech Challenge - Fase 3 | Pós-Graduação em Data Analytics**

---

### 🎯 1. Objetivo e Perguntas Estratégicas de Negócio
Este projeto processa a **SOT (Source of Truth)** oficial do *State of Data Brasil* utilizando **Apache Spark (PySpark)** para engenharia de dados em larga escala, gerando **7 SPECS Analíticas** otimizadas em formato Parquet. Em seguida, conecta o motor analítico **DuckDB** e dashboards executivos interativos em **Plotly** para responder às 7 perguntas estratégicas de negócio:

1. **Como está estruturado o mercado brasileiro de Dados?**
2. **Quais perfis profissionais são mais valorizados pelo mercado?**
3. **Qual é o cenário de diversidade de gênero nas carreiras de dados?**
4. **Quais tecnologias apresentam maior adoção entre os profissionais?**
5. **Qual é o índice de adoção de Inteligência Artificial e seu impacto?**
6. **Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?**
7. **Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?**

---

### 🏛️ 2. Arquitetura da Camada Analítica com PySpark (7 SPECS)
```text
dados/base_consolidada.parquet (SOT Auditável - 14.005 respondentes)
       │
       ▼ [Engenharia de Dados & Transformação com PySpark DataFrame API]
       ├── 1. spec_respondentes.parquet                  -> Perfil demográfico completo, macro-cargos e salários
       ├── 2. spec_adocao_tecnologias.parquet            -> Stack tecnológico detalhado (Linguagens, Cloud, Bancos, ETL)
       ├── 3. spec_adocao_ia.parquet                     -> Adoção de IA (Uso Individual vs Corporativo vs Barreiras)
       ├── 4. spec_diversidade_carreira.parquet          -> Métricas agregadas de equidade, efeito funil e pay gap
       ├── 5. spec_segmentacao_negocio.parquet           -> Clusters executivos para planejamento de remuneração e IA
       ├── 6. spec_dinamica_trabalho_satisfacao.parquet  -> Modelo de trabalho atual vs ideal e retenção de talentos
       └── 7. spec_estrutura_times_empresa.parquet       -> Composição e papéis das equipes de dados nas empresas
       │
       ▼
Motor SQL DuckDB + Dashboards Executivos Plotly
```


In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração automática do JAVA_HOME para execução do PySpark
if 'JAVA_HOME' not in os.environ:
    java_candidate = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot'
    if os.path.exists(java_candidate):
        os.environ['JAVA_HOME'] = java_candidate
        os.environ['PATH'] = os.path.join(java_candidate, 'bin') + os.pathsep + os.environ.get('PATH', '')

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd
import numpy as np

# Configuração de caminhos
BASE_PROJETO = Path.cwd()
ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'

if not ORIGEM_SOT.exists():
    ORIGEM_SOT = BASE_PROJETO / 'projeto' / 'fase_3_data_analytics' / 'dados' / 'base_consolidada.parquet'
    SAIDA_SPECS = BASE_PROJETO / 'projeto' / 'fase_3_data_analytics' / 'dados' / 'bases_analiticas'

if not ORIGEM_SOT.exists():
    ORIGEM_SOT = Path('D:/d/pos/Fase 3/projeto/fase_3_data_analytics/dados/base_consolidada.parquet')
    SAIDA_SPECS = Path('D:/d/pos/Fase 3/projeto/fase_3_data_analytics/dados/bases_analiticas')

assert ORIGEM_SOT.exists(), f'SOT não encontrada: {ORIGEM_SOT}'
SAIDA_SPECS.mkdir(parents=True, exist_ok=True)

# 1. Inicializando SparkSession
spark = SparkSession.builder \
    .appName("TechChallenge_Fase3_SPECS") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 2. Leitura da SOT via PySpark
df_spark = spark.read.parquet(str(ORIGEM_SOT))

# 3. Engenharia de Features & Chave de Respondente com PySpark
df_spark = df_spark.withColumn("id_str", F.col("id").cast(T.StringType())) \
                   .withColumn("ano_pesquisa", F.col("ano_pesquisa").cast(T.IntegerType())) \
                   .withColumn("respondente_key", F.concat_ws("_", F.col("ano_pesquisa"), F.col("id_str")))

# 4. Normalização Salarial Numérica (PySpark when/otherwise)
df_spark = df_spark.withColumn(
    "salario_estimado_num",
    F.when(F.col("faixa_salarial") == "Menos de R$ 1.000/mês", 1000.0)
     .when(F.col("faixa_salarial").isin("de R$ 101/mês a R$ 2.000/mês", "de R$ 1.001/mês a R$ 2.000/mês"), 1500.0)
     .when(F.col("faixa_salarial") == "de R$ 2.001/mês a R$ 3.000/mês", 2500.0)
     .when(F.col("faixa_salarial") == "de R$ 3.001/mês a R$ 4.000/mês", 3500.0)
     .when(F.col("faixa_salarial") == "de R$ 4.001/mês a R$ 6.000/mês", 5000.0)
     .when(F.col("faixa_salarial") == "de R$ 6.001/mês a R$ 8.000/mês", 7000.0)
     .when(F.col("faixa_salarial") == "de R$ 8.001/mês a R$ 12.000/mês", 10000.0)
     .when(F.col("faixa_salarial") == "de R$ 12.001/mês a R$ 16.000/mês", 14000.0)
     .when(F.col("faixa_salarial") == "de R$ 16.001/mês a R$ 20.000/mês", 18000.0)
     .when(F.col("faixa_salarial") == "de R$ 20.001/mês a R$ 25.000/mês", 22500.0)
     .when(F.col("faixa_salarial").isin("de R$ 25.001/mês a R$ 3000/mês", "de R$ 25.001/mês a R$ 30.000/mês"), 27500.0)
     .when(F.col("faixa_salarial") == "de R$ 30.001/mês a R$ 40.000/mês", 35000.0)
     .when(F.col("faixa_salarial") == "Acima de R$ 40.001/mês", 45000.0)
     .otherwise(F.lit(None).cast(T.DoubleType()))
).withColumn(
    "ordem_salarial",
    F.when(F.col("faixa_salarial") == "Menos de R$ 1.000/mês", 1)
     .when(F.col("faixa_salarial").isin("de R$ 101/mês a R$ 2.000/mês", "de R$ 1.001/mês a R$ 2.000/mês"), 2)
     .when(F.col("faixa_salarial") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
     .when(F.col("faixa_salarial") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
     .when(F.col("faixa_salarial") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
     .when(F.col("faixa_salarial") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
     .when(F.col("faixa_salarial") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
     .when(F.col("faixa_salarial") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
     .when(F.col("faixa_salarial") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
     .when(F.col("faixa_salarial") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
     .when(F.col("faixa_salarial").isin("de R$ 25.001/mês a R$ 3000/mês", "de R$ 25.001/mês a R$ 30.000/mês"), 11)
     .when(F.col("faixa_salarial") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
     .when(F.col("faixa_salarial") == "Acima de R$ 40.001/mês", 13)
     .otherwise(F.lit(None).cast(T.IntegerType()))
)

# 5. Padronização de Macro-Cargos via PySpark
df_spark = df_spark.withColumn(
    "macro_cargo",
    F.when(F.col("cargo_atual").isNull(), "Não informado / Em transição")
     .when(F.col("cargo_atual").rlike("(?i)Engenheiro de Dados|Arquiteto de Dados|Analytics Engineer"), "Engenharia & Arquitetura de Dados")
     .when(F.col("cargo_atual").rlike("(?i)Cientista de Dados|Estatístico|Economista"), "Ciência de Dados")
     .when(F.col("cargo_atual").rlike("(?i)Analista de Dados|Analista de BI|Inteligência de Mercado"), "Análise de Dados & BI")
     .when(F.col("cargo_atual").rlike("(?i)Machine Learning|ML Engineer|AI Engineer"), "Machine Learning & IA")
     .when(F.col("cargo_atual").rlike("(?i)Analista de Negócios|Business Analyst|Product Manager|PM/APM/DPM"), "Negócios & Gestão de Produto")
     .when(F.col("cargo_atual").rlike("(?i)DBA|Administrador de Banco"), "DBA & Infraestrutura")
     .when(F.col("cargo_atual").rlike("(?i)Desenvolvedor|Engenheiro de Software|Analista de Sistemas|Outras Engenharias|Suporte"), "Engenharia de Software / Outras Engenharias")
     .when(F.col("cargo_atual").rlike("(?i)Professor|Pesquisador"), "Academia & Pesquisa")
     .otherwise("Outros")
)

# 6. Normalização de Modelos de Trabalho via PySpark
df_spark = df_spark.withColumn(
    "modelo_trabalho_resumido",
    F.when(F.col("modelo_trabalho_atual").isNull(), "Não informado")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)100% remoto|remoto"), "Remoto")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)flexível|flexivel"), "Híbrido Flexível")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)fixo|dias fixos"), "Híbrido Fixo")
     .when(F.col("modelo_trabalho_atual").rlike("(?i)presencial"), "Presencial")
     .otherwise(F.col("modelo_trabalho_atual"))
).withColumn(
    "modelo_ideal_resumido",
    F.when(F.col("modelo_trabalho_ideal").isNull(), "Não informado")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)100% remoto|remoto"), "Remoto")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)flexível|flexivel"), "Híbrido Flexível")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)fixo|dias fixos"), "Híbrido Fixo")
     .when(F.col("modelo_trabalho_ideal").rlike("(?i)presencial"), "Presencial")
     .otherwise(F.col("modelo_trabalho_ideal"))
)

# --- GERAÇÃO DAS 7 SPECS ANALÍTICAS ---

# SPEC 1: spec_respondentes
dimensoes_resp = [
    'respondente_key', 'id', 'ano_pesquisa', 'genero', 'cor_raca_etnia', 'faixa_idade',
    'estado_onde_mora', 'uf_onde_mora', 'regiao_onde_mora', 'nivel_de_ensino',
    'area_de_formacao', 'setor', 'cargo_atual', 'macro_cargo', 'nivel',
    'faixa_salarial', 'salario_estimado_num', 'ordem_salarial',
    'tempo_experiencia_dados', 'oportunidade_buscada',
    'modelo_trabalho_atual', 'modelo_trabalho_resumido', 'modelo_trabalho_ideal', 'modelo_ideal_resumido'
]
cols_existentes_resp = [c for c in dimensoes_resp if c in df_spark.columns]
spark_spec_respondentes = df_spark.select(cols_existentes_resp).dropDuplicates(['respondente_key'])
spark_spec_respondentes.toPandas().to_parquet(SAIDA_SPECS / 'spec_respondentes.parquet', index=False)

# SPEC 2: spec_adocao_tecnologias
familias_tecnologia = {
    'linguagem': ['sql', 'r', 'python', 'c_cpp_csharp', 'dotnet', 'java', 'julia', 'sas_stata', 'visual_basic_vba', 'scala', 'matlab', 'rust', 'php', 'javascript'],
    'banco_ou_plataforma': ['mysql', 'oracle', 'sql_server', 'amazon_aurora_rds', 'dynamodb', 'coachdb', 'cassandra', 'mongodb', 'mariadb', 'datomic', 's3', 'postgresql', 'elasticsearch', 'db2', 'microsoft_access', 'sqlite', 'sybase', 'firebase', 'vertica', 'redis', 'neo4j', 'google_bigquery', 'google_firestore', 'amazon_redshift', 'amazon_athena', 'snowflake', 'databricks', 'hbase', 'presto', 'splunk', 'sap_hana', 'hive', 'firebird'],
    'cloud': ['aws_cloud', 'gcp_cloud', 'azure_cloud', 'oracle_cloud', 'ibm', 'cloud_propria'],
    'etl': ['scripts_python', 'sql_stored_procedures', 'apache_airflow', 'apache_nifi', 'luigi', 'aws_glue', 'talend', 'pentaho', 'alteryx', 'stitch', 'fivetran', 'google_dataflow', 'oracle_data_integrator', 'ibm_datastage', 'sap_bw_etl', 'sql_server_integration_services_ssis', 'sas_data_integration', 'qlik_sense', 'knime', 'databricks_etl']
}
nomes_formatados = {
    'sql': 'SQL', 'r': 'R', 'python': 'Python', 'c_cpp_csharp': 'C/C++/C#', 'dotnet': '.NET', 'java': 'Java', 'julia': 'Julia', 'sas_stata': 'SAS/Stata', 'visual_basic_vba': 'VBA', 'scala': 'Scala', 'matlab': 'Matlab', 'rust': 'Rust', 'php': 'PHP', 'javascript': 'JavaScript', 'aws_cloud': 'AWS', 'gcp_cloud': 'Google Cloud (GCP)', 'azure_cloud': 'Azure', 'oracle_cloud': 'Oracle Cloud', 'ibm': 'IBM Cloud', 'cloud_propria': 'Cloud Própria / On-Premise', 'google_bigquery': 'Google BigQuery', 'amazon_redshift': 'Amazon Redshift', 'amazon_athena': 'Amazon Athena', 'amazon_aurora_rds': 'Amazon Aurora/RDS', 'sql_server': 'SQL Server', 'postgresql': 'PostgreSQL', 'mysql': 'MySQL', 'sqlite': 'SQLite', 'mongodb': 'MongoDB', 'snowflake': 'Snowflake', 'databricks': 'Databricks', 'scripts_python': 'Scripts Python', 'sql_stored_procedures': 'Stored Procedures SQL', 'apache_airflow': 'Apache Airflow', 'apache_nifi': 'Apache NiFi', 'aws_glue': 'AWS Glue', 'databricks_etl': 'Databricks (ETL/Workflows)', 'google_dataflow': 'Google Dataflow', 'sql_server_integration_services_ssis': 'SSIS', 'coachdb': 'CouchDB'
}
cols_contexto_tech = [c for c in ['respondente_key', 'id', 'ano_pesquisa', 'genero', 'regiao_onde_mora', 'cargo_atual', 'macro_cargo', 'nivel', 'tempo_experiencia_dados', 'faixa_salarial', 'salario_estimado_num', 'modelo_trabalho_resumido', 'setor'] if c in df_spark.columns]
tech_dfs = []
for familia, cols in familias_tecnologia.items():
    for col in cols:
        if col in df_spark.columns:
            nome_tech = nomes_formatados.get(col, col.replace('_cloud', '').replace('_etl', '').replace('_', ' ').strip().title())
            sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                          .select(cols_contexto_tech) \
                          .withColumn("familia", F.lit(familia)) \
                          .withColumn("tecnologia_id", F.lit(col)) \
                          .withColumn("tecnologia", F.lit(nome_tech)) \
                          .withColumn("tecnologia_principal", F.lit(False))
            tech_dfs.append(sub)

spark_spec_tech = tech_dfs[0]
for tdf in tech_dfs[1:]:
    spark_spec_tech = spark_spec_tech.unionByName(tdf)
spark_spec_tech = spark_spec_tech.dropDuplicates(['respondente_key', 'familia', 'tecnologia'])

if 'linguagem_principal' in df_spark.columns:
    df_ling = df_spark.select('respondente_key', 'linguagem_principal').filter(F.col('linguagem_principal').isNotNull()) \
                      .withColumn('ling_lower', F.lower(F.trim(F.col('linguagem_principal'))))
    spark_spec_tech = spark_spec_tech.join(df_ling, on='respondente_key', how='left') \
                                     .withColumn('tecnologia_principal', (F.col('familia') == 'linguagem') & (F.lower(F.col('tecnologia')) == F.col('ling_lower'))) \
                                     .drop('ling_lower', 'linguagem_principal')
spark_spec_tech.toPandas().to_parquet(SAIDA_SPECS / 'spec_adocao_tecnologias.parquet', index=False)

# SPEC 3: spec_adocao_ia
dicionario_ia = {
    'ia_solucoes_gratuitas': ('Uso Individual / Produtividade', 'Uso de Soluções Gratuitas (ChatGPT, Gemini, etc.)', 'Uso Ativo'),
    'ia_paga_usuario': ('Uso Individual / Produtividade', 'Assinatura Paga pelo Próprio Profissional', 'Uso Ativo'),
    'ia_paga_empresa': ('Uso Individual / Produtividade', 'Assinatura Paga pela Empresa', 'Uso Ativo'),
    'usa_copilot': ('Uso Individual / Produtividade', 'Uso de Assistente de Código (Copilot, Cursor, etc.)', 'Uso Ativo'),
    'ia_independente_descentralizada': ('Uso Individual / Produtividade', 'Uso Descentralizado / Autônomo por Colaboradores', 'Uso Ativo'),
    'ia_direcionamento_centralizado': ('Uso Organizacional / Negócio', 'Direcionamento Centralizado Corporativo', 'Uso Ativo'),
    'desenvolvedores_copilot': ('Uso Organizacional / Negócio', 'Equipes de Dev/Dados com Ferramentas de IA', 'Uso Ativo'),
    'ia_produtos_internos': ('Uso Organizacional / Negócio', 'IA Aplicada em Eficiência e Processos Internos', 'Uso Ativo'),
    'ia_produtos_externos': ('Uso Organizacional / Negócio', 'IA Integrada a Produtos e Serviços para Clientes', 'Uso Ativo'),
    'ia_principal_frente_negocio': ('Uso Organizacional / Negócio', 'IA como Frente Central de Diferenciação do Negócio', 'Uso Ativo'),
    'ia_nao_prioridade': ('Postura Organizacional', 'IA Não é Prioridade Estratégica no Momento', 'Postura'),
    'falta_compreensao_casos_uso': ('Barreira / Desafio', 'Falta de Compreensão dos Casos de Uso', 'Barreira'),
    'falta_confiabilidade_saidas': ('Barreira / Desafio', 'Falta de Confiabilidade / Alucinações nas Saídas', 'Barreira'),
    'incerteza_regulamentacao': ('Barreira / Desafio', 'Incertezas Regulatórias e Jurídicas', 'Barreira'),
    'preocupacoes_seguranca_privacidade': ('Barreira / Desafio', 'Preocupações com Segurança e Privacidade de Dados', 'Barreira'),
    'roi_nao_comprovado_ia': ('Barreira / Desafio', 'Retorno Financeiro (ROI) Não Comprovado', 'Barreira'),
    'dados_empresa_nao_prontos_ia': ('Barreira / Desafio', 'Dados Despreparados / Falta de Governança', 'Barreira'),
    'falta_expertise_recursos': ('Barreira / Desafio', 'Falta de Expertise Técnica e Mão de Obra', 'Barreira'),
    'alta_direcao_nao_ve_valor': ('Barreira / Desafio', 'Alta Direção Não Vê Valor Imediato', 'Barreira'),
    'preocupacoes_propriedade_intelectual': ('Barreira / Desafio', 'Preocupações com Propriedade Intelectual', 'Barreira')
}
cols_contexto_ia = [c for c in ['respondente_key', 'id', 'ano_pesquisa', 'genero', 'regiao_onde_mora', 'cargo_atual', 'macro_cargo', 'nivel', 'tempo_experiencia_dados', 'faixa_salarial', 'salario_estimado_num', 'modelo_trabalho_resumido', 'setor'] if c in df_spark.columns]
ia_dfs = []
for col, (categoria, label, tipo_registro) in dicionario_ia.items():
    if col in df_spark.columns:
        sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                      .select(cols_contexto_ia) \
                      .withColumn("indicador_ia_id", F.lit(col)) \
                      .withColumn("categoria_ia", F.lit(categoria)) \
                      .withColumn("indicador_ia_label", F.lit(label)) \
                      .withColumn("tipo_registro", F.lit(tipo_registro)) \
                      .withColumn("valor", F.lit(1))
        ia_dfs.append(sub)

spark_spec_ia = ia_dfs[0]
for idf in ia_dfs[1:]:
    spark_spec_ia = spark_spec_ia.unionByName(idf)
spark_spec_ia = spark_spec_ia.dropDuplicates(['respondente_key', 'indicador_ia_id'])
spark_spec_ia.toPandas().to_parquet(SAIDA_SPECS / 'spec_adocao_ia.parquet', index=False)

# SPEC 4: spec_diversidade_carreira
spark_spec_div = df_spark.filter(F.col("genero").isNotNull()) \
                         .groupBy("ano_pesquisa", "macro_cargo", "nivel", "genero", "cor_raca_etnia") \
                         .agg(
                             F.countDistinct("respondente_key").alias("respondentes"),
                             F.round(F.avg("salario_estimado_num"), 2).alias("salario_medio"),
                             F.expr("percentile_approx(salario_estimado_num, 0.5)").alias("salario_mediano"),
                             F.round(100.0 * F.avg(F.when(F.col("modelo_trabalho_resumido") == "Remoto", 1.0).otherwise(0.0)), 1).alias("remoto_pct")
                         )
spark_spec_div.toPandas().to_parquet(SAIDA_SPECS / 'spec_diversidade_carreira.parquet', index=False)

# SPEC 5: spec_segmentacao_negocio
adotantes_ia_df = spark_spec_ia.filter(F.col("tipo_registro") == "Uso Ativo").select("respondente_key").distinct().withColumn("usa_ia_ativo", F.lit(1.0))
df_resp_ia = df_spark.join(adotantes_ia_df, on="respondente_key", how="left").fillna({"usa_ia_ativo": 0.0}).withColumn("is_mulher", F.when(F.col("genero") == "Feminino", 1.0).otherwise(0.0))
spark_spec_seg = df_resp_ia.groupBy("ano_pesquisa", "macro_cargo", "nivel", "regiao_onde_mora", "modelo_trabalho_resumido") \
                           .agg(
                               F.countDistinct("respondente_key").alias("total_respondentes"),
                               F.round(F.avg("salario_estimado_num"), 2).alias("salario_medio_estimado"),
                               F.expr("percentile_approx(salario_estimado_num, 0.5)").alias("salario_mediano_estimado"),
                               F.round(100.0 * F.avg("usa_ia_ativo"), 1).alias("pct_adotantes_ia"),
                               F.round(100.0 * F.avg("is_mulher"), 1).alias("pct_mulheres")
                           )
spark_spec_seg.toPandas().to_parquet(SAIDA_SPECS / 'spec_segmentacao_negocio.parquet', index=False)

# SPEC 6: spec_dinamica_trabalho_satisfacao
spark_spec_trab = df_spark.withColumn(
    "status_alinhamento",
    F.when(F.col("modelo_trabalho_resumido") == "Não informado", "Não informado")
     .when(F.col("modelo_ideal_resumido") == "Não informado", "Não informado")
     .when(F.col("modelo_trabalho_resumido") == F.col("modelo_ideal_resumido"), "Alinhado (Modelo Satisfeito)")
     .when(F.col("modelo_trabalho_resumido").isin("Presencial", "Híbrido Fixo") & F.col("modelo_ideal_resumido").isin("Remoto", "Híbrido Flexível"), "Deseja Mais Flexibilidade / Remoto")
     .when((F.col("modelo_trabalho_resumido") == "Remoto") & F.col("modelo_ideal_resumido").isin("Presencial", "Híbrido Flexível", "Híbrido Fixo"), "Deseja Mais Presencial / Escritório")
     .otherwise("Outro Descompasso")
).select(
    'respondente_key', 'id', 'ano_pesquisa', 'macro_cargo', 'nivel', 'regiao_onde_mora',
    'setor', 'modelo_trabalho_resumido', 'modelo_ideal_resumido', 'status_alinhamento',
    'oportunidade_buscada', 'salario_estimado_num', 'tempo_experiencia_dados'
)
spark_spec_trab.toPandas().to_parquet(SAIDA_SPECS / 'spec_dinamica_trabalho_satisfacao.parquet', index=False)

# SPEC 7: spec_estrutura_times_empresa
papeis_empresa = {
    'analytics_engineer': 'Analytics Engineer', 'engenheiro_de_dados': 'Engenheiro de Dados', 'analisa_de_dados': 'Analista de Dados', 'cientista_de_dados': 'Cientista de Dados', 'database_administrator': 'DBA / Administrador de Banco', 'analista_de_business': 'Analista de Negócios (BI/BA)', 'arquiteto_de_dados': 'Arquiteto de Dados', 'product_manager': 'Data Product Manager', 'business_analyst': 'Business Analyst', 'ml_engineer': 'Machine Learning Engineer'
}
time_dfs = []
cols_contexto_time = ['respondente_key', 'id', 'ano_pesquisa', 'setor', 'regiao_onde_mora', 'macro_cargo']
for col, label in papeis_empresa.items():
    if col in df_spark.columns:
        sub = df_spark.filter(F.col(col).isin("1", "1.0", "true", "True", "SIM", "Sim", "sim", "x", "X")) \
                      .select(cols_contexto_time) \
                      .withColumn("papel_na_empresa", F.lit(label)) \
                      .withColumn("papel_id", F.lit(col)) \
                      .withColumn("presente_na_empresa", F.lit(True))
        time_dfs.append(sub)

if len(time_dfs) > 0:
    spark_spec_times = time_dfs[0]
    for tdf in time_dfs[1:]:
        spark_spec_times = spark_spec_times.unionByName(tdf)
    spark_spec_times = spark_spec_times.dropDuplicates(['respondente_key', 'papel_na_empresa'])
    spark_spec_times.toPandas().to_parquet(SAIDA_SPECS / 'spec_estrutura_times_empresa.parquet', index=False)

spark.stop()

print(f"✓ SOT Consumida com sucesso via PySpark: {ORIGEM_SOT}")
print(f"✓ 7 SPECS Analíticas geradas em: {SAIDA_SPECS}")
for f in sorted(SAIDA_SPECS.glob('*.parquet')):
    p = pd.read_parquet(f)
    print(f"  - {f.name:<45} | Shape: {p.shape}")


In [ ]:
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlglot

con = duckdb.connect()

# Registra as 7 SPECS no DuckDB como Views
for f in SAIDA_SPECS.glob('*.parquet'):
    nome = f.stem
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_parquet('{f.as_posix()}')")

def executar_sql(query):
    query_validada = sqlglot.parse_one(query, read="duckdb")
    return con.sql(query_validada.sql(dialect="duckdb")).df()

print("✓ Motor SQL DuckDB configurado e conectado a todas as 7 SPECS!")


## 🎯 Perguntas 1 & 2: Estrutura do Mercado & Perfis mais Valorizados
* **P1. Como está estruturado o mercado brasileiro de Dados?**
  * O mercado é liderado numericamente por **Análise de Dados & BI** (25.8%), seguido por **Engenharia & Arquitetura de Dados** (16.3%) e **Ciência de Dados** (13.2%).
  * Nas empresas, as funções mais presentes nas equipes de dados são **Analista de BI/Dados** (presente em 78.4% das organizações), **Engenheiro de Dados** (67.2%) e **Cientista de Dados** (52.1%).
* **P2. Quais perfis profissionais são mais valorizados pelo mercado?**
  * **Especialistas em Machine Learning & IA** recebem a maior remuneração média de mercado (R$ 16.186), seguidos por **Engenharia de Dados** (R$ 12.603) e **Ciência de Dados** (R$ 11.560).
  * No corte de stack tecnológico, o domínio de plataformas analíticas modernas escaláveis (**Snowflake, Databricks, Redis, Scala, AWS Glue**) correlaciona-se diretamente com as maiores faixas salariais.


In [ ]:

# Consulta SQL: Estrutura de Mercado e Remuneração Média
df_cargos = executar_sql("""
    SELECT macro_cargo,
           COUNT(*) AS total_profissionais,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM spec_respondentes), 1) AS pct_mercado,
           ROUND(AVG(salario_estimado_num), 0) AS salario_medio
    FROM spec_respondentes
    WHERE macro_cargo NOT IN ('Não informado / Em transição', 'Outros')
    GROUP BY 1
    ORDER BY total_profissionais ASC
""")

# Consulta SQL: Remuneração por Nível e Cargo
df_nivel_cargo = executar_sql("""
    SELECT macro_cargo, nivel,
           ROUND(AVG(salario_estimado_num), 0) AS salario_medio,
           COUNT(*) AS total
    FROM spec_respondentes
    WHERE salario_estimado_num IS NOT NULL 
      AND nivel IN ('Júnior', 'Pleno', 'Sênior', 'Especialista/Staff+')
      AND macro_cargo IN ('Engenharia & Arquitetura de Dados', 'Ciência de Dados', 'Análise de Dados & BI', 'Machine Learning & IA')
    GROUP BY 1, 2
    ORDER BY CASE nivel WHEN 'Júnior' THEN 1 WHEN 'Pleno' THEN 2 WHEN 'Sênior' THEN 3 WHEN 'Especialista/Staff+' THEN 4 END
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Distribuição de Profissionais por Macro-Cargo</b>', '<b>Remuneração Média (R$) por Senioridade</b>'),
    horizontal_spacing=0.12
)

# Gráfico 1: Barras horizontais de volume de profissionais
fig.add_trace(go.Bar(
    x=df_cargos['total_profissionais'], y=df_cargos['macro_cargo'],
    orientation='h', marker_color='#1f77b4',
    text=df_cargos.apply(lambda r: f"{r['total_profissionais']:,} ({r['pct_mercado']}%)", axis=1),
    textposition='outside', name='Volume'
), row=1, col=1)

# Gráfico 2: Barras agrupadas de salário por nível
cores = {'Engenharia & Arquitetura de Dados': '#2ca02c', 'Ciência de Dados': '#9467bd', 'Machine Learning & IA': '#ff7f0e', 'Análise de Dados & BI': '#17becf'}
for cargo in df_nivel_cargo['macro_cargo'].unique():
    sub = df_nivel_cargo[df_nivel_cargo['macro_cargo'] == cargo]
    fig.add_trace(go.Bar(
        x=sub['nivel'], y=sub['salario_medio'],
        name=cargo, marker_color=cores.get(cargo, '#333333'),
        text=sub['salario_medio'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
    ), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=450,
    title={'text': '<b>Estrutura do Mercado e Valorização Salarial em Dados</b>', 'font': {'size': 18}},
    barmode='group', showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)
fig.show()


## 🎯 Pergunta 3: Diversidade de Gênero, Efeito Funil e Pay Gap
* **P3. Qual é o cenário de diversidade de gênero nas carreiras de dados?**
  * **Sub-representação Geral**: Mulheres compõem apenas **23.6%** do mercado de dados brasileiro.
  * **Efeito Funil ("Teto de Vidro")**: A participação feminina declina progressivamente conforme a senioridade avança: **27.2% no nível Júnior**, **24.8% no Pleno**, **20.5% no Sênior** e apenas **16.2% em Especialista/Staff+**.
  * **Disparidade Salarial (Pay Gap)**: A diferença salarial homem vs mulher se acentua no topo da carreira: no nível Sênior, homens ganham em média **15.6% a mais**, e no nível Especialista/Staff+, **14.9% a mais**.


In [ ]:
df_funil = executar_sql("""
    SELECT nivel, 
           COUNT(*) AS total,
           ROUND(100.0 * SUM(CASE WHEN genero = 'Feminino' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_mulheres,
           ROUND(100.0 * SUM(CASE WHEN genero = 'Masculino' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_homens,
           ROUND(AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END), 0) AS sal_homens,
           ROUND(AVG(CASE WHEN genero = 'Feminino' THEN salario_estimado_num END), 0) AS sal_mulheres,
           ROUND(100.0 * (AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END) - AVG(CASE WHEN genero = 'Feminino' THEN salario_estimado_num END)) / AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END), 1) AS pay_gap_pct
    FROM spec_respondentes 
    WHERE nivel IN ('Júnior', 'Pleno', 'Sênior', 'Especialista/Staff+')
      AND genero IN ('Masculino', 'Feminino')
    GROUP BY nivel 
    ORDER BY CASE nivel WHEN 'Júnior' THEN 1 WHEN 'Pleno' THEN 2 WHEN 'Sênior' THEN 3 WHEN 'Especialista/Staff+' THEN 4 END
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Funil de Representatividade Feminina por Senioridade</b>', '<b>Comparativo Salarial Médio (Homens vs Mulheres)</b>'),
    horizontal_spacing=0.12
)

# Gráfico 1: Funil % Mulheres
fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['pct_mulheres'],
    marker_color='#e377c2', name='% Mulheres',
    text=df_funil['pct_mulheres'].apply(lambda v: f"{v}%"), textposition='outside'
), row=1, col=1)

# Gráfico 2: Pay Gap Homens vs Mulheres
fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['sal_homens'],
    name='Homens (R$)', marker_color='#1f77b4',
    text=df_funil['sal_homens'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
), row=1, col=2)

fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['sal_mulheres'],
    name='Mulheres (R$)', marker_color='#e377c2',
    text=df_funil['sal_mulheres'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=450,
    title={'text': '<b>Cenário de Diversidade de Gênero e Equidade Salarial</b>', 'font': {'size': 18}},
    barmode='group', showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)
fig.update_yaxes(range=[0, 35], title='% Representação Feminina', row=1, col=1)
fig.update_yaxes(title='Salário Médio (R$)', row=1, col=2)
fig.show()


## 🎯 Pergunta 4: Adoção Tecnológica (Linguagens, Cloud, Bancos e ETL)
* **P4. Quais tecnologias apresentam maior adoção entre os profissionais?**
  * **Linguagens**: **SQL (57.6%)** e **Python (54.9%)** formam o duopólio absoluto do mercado brasileiro. O R mantém 7.8% de nicho acadêmico/estatístico.
  * **Cloud Providers**: **AWS lidera com 26.4%**, seguida de perto por **Azure (24.0%)** e **Google Cloud (20.3%)**, demonstrando um mercado corporativo fortemente multi-cloud.
  * **Bancos & Data Platforms**: **PostgreSQL (21.2%)** e **SQL Server (21.2%)** dominam o transactional/relacional, enquanto **Databricks (18.7%)**, **BigQuery (15.5%)** e **Snowflake** lideram o moderno stack analítico.


In [ ]:
df_tech = executar_sql("""
    WITH ranked AS (
        SELECT familia, tecnologia,
               COUNT(DISTINCT respondente_key) AS adotantes,
               ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS pct_mercado,
               ROW_NUMBER() OVER(PARTITION BY familia ORDER BY COUNT(DISTINCT respondente_key) DESC) AS rnk
        FROM spec_adocao_tecnologias
        GROUP BY 1, 2
    )
    SELECT familia, tecnologia, adotantes, pct_mercado
    FROM ranked WHERE rnk <= 5
    ORDER BY familia, pct_mercado DESC
""")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('<b>Linguagens de Programação</b>', '<b>Cloud Providers</b>',
                    '<b>Bancos de Dados & Data Platforms</b>', '<b>Ferramentas de ETL & Orquestração</b>'),
    vertical_spacing=0.15, horizontal_spacing=0.12
)

def add_family_bar(familia, row, col, color):
    sub = df_tech[df_tech['familia'] == familia].sort_values('pct_mercado', ascending=True)
    fig.add_trace(go.Bar(
        x=sub['pct_mercado'], y=sub['tecnologia'],
        orientation='h', marker_color=color,
        text=sub['pct_mercado'].apply(lambda v: f"{v}%"), textposition='outside'
    ), row=row, col=col)

add_family_bar('linguagem', 1, 1, '#1f77b4')
add_family_bar('cloud', 1, 2, '#ff7f0e')
add_family_bar('banco_ou_plataforma', 2, 1, '#2ca02c')
add_family_bar('etl', 2, 2, '#d62728')

fig.update_layout(
    template='plotly_white', height=650, showlegend=False,
    title={'text': '<b>Stack Tecnológico Líder no Mercado Brasileiro de Dados</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 70])
fig.show()


## 🎯 Perguntas 5 & 6: Adoção de IA e Satisfação com Modelos de Trabalho
* **P5. Qual é o índice de adoção de Inteligência Artificial e seu impacto?**
  * **64.3% dos profissionais** já adotam soluções de IA ativamente em seu fluxo de trabalho.
  * O uso individual/gratuito predomina como porta de entrada (35.6%), mas a aplicação estruturada em **eficiência de processos internos (17.5%)** e **produtos para clientes finais (16.0%)** já atinge escala expressiva.
* **P6. Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?**
  * **Modelo de Trabalho**: Profissionais em regime **100% Remoto (71.2%)** e **Híbrido Flexível (66.5%)** têm taxas de adoção de IA substancialmente maiores que no **Presencial (51.8%)**.
  * **Descompasso Trabalho Atual vs Ideal**: Mais de **48% dos profissionais presenciais ou híbridos fixos desejam maior flexibilidade/remoto**, representando o principal vetor de atrito e turnover no mercado.


In [ ]:
df_ia_rotina = executar_sql("""
    SELECT COALESCE(modelo_trabalho_resumido, 'Não informado') AS modelo,
           COUNT(DISTINCT r.respondente_key) AS total,
           COUNT(DISTINCT a.respondente_key) AS adotantes,
           ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) AS pct_adocao
    FROM spec_respondentes r
    LEFT JOIN (SELECT DISTINCT respondente_key FROM spec_adocao_ia WHERE tipo_registro = 'Uso Ativo') a 
      ON r.respondente_key = a.respondente_key
    WHERE modelo_trabalho_resumido != 'Não informado'
    GROUP BY 1 ORDER BY pct_adocao ASC
""")

df_alinhamento = executar_sql("""
    SELECT status_alinhamento,
           COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM spec_dinamica_trabalho_satisfacao WHERE status_alinhamento != 'Não informado'), 1) AS pct
    FROM spec_dinamica_trabalho_satisfacao
    WHERE status_alinhamento != 'Não informado'
    GROUP BY 1 ORDER BY total ASC
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Taxa de Adoção de IA por Regime de Trabalho</b>', '<b>Alinhamento entre Modelo de Trabalho Atual vs Ideal</b>'),
    horizontal_spacing=0.12
)

fig.add_trace(go.Bar(
    x=df_ia_rotina['pct_adocao'], y=df_ia_rotina['modelo'],
    orientation='h', marker_color='#0b7285',
    text=df_ia_rotina.apply(lambda r: f"{r['pct_adocao']}% (n={r['total']:,})", axis=1), textposition='inside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_alinhamento['pct'], y=df_alinhamento['status_alinhamento'],
    orientation='h', marker_color='#845ef7',
    text=df_alinhamento.apply(lambda r: f"{r['pct']}% (n={r['total']:,})", axis=1), textposition='inside'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=450, showlegend=False,
    title={'text': '<b>Dinâmica de Trabalho, Adoção Tecnológica e Retenção de Talentos</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 100], title='% Adoção de IA', row=1, col=1)
fig.update_xaxes(range=[0, 100], title='% dos Profissionais', row=1, col=2)
fig.show()


## 🎯 Pergunta 7: Oportunidades, Desafios e Plano de Ação para Investimento
* **P7. Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?**
  * **Top 3 Desafios / Barreiras de Negócio**:
    1. **Falta de Expertise Técnica e Mão de Obra (36.8%)**;
    2. **Falta de Compreensão dos Casos de Uso Reais (36.0%)**;
    3. **Dados da Empresa Despreparados / Falta de Governança (33.1%)**.
  * **A Grande Oportunidade**: A barreira número 1 **não é financeira ou regulatória**, mas de **Prontidão de Dados e Capacitação**. Empresas que investem em estruturar sua base de dados (Modern Data Stack, Governança, Catálogo) e capacitar seu time em SQL, Python e Cloud conseguem destravar ROI tangível em IA com alta velocidade.


In [ ]:
df_barreiras = executar_sql("""
    SELECT indicador_ia_label AS barreira,
           COUNT(DISTINCT respondente_key) AS ocorrencias,
           ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_adocao_ia WHERE tipo_registro = 'Barreira'), 1) AS pct_barreiras
    FROM spec_adocao_ia
    WHERE tipo_registro = 'Barreira'
    GROUP BY 1 ORDER BY ocorrencias ASC
""")

resumo_diretoria = pd.DataFrame([
    {"Pilar Estratégico": "Pessoas & Diversidade", 
     "Diagnóstico dos Dados": "Mulheres representam 23.6% do mercado com teto de vidro nos níveis seniores (16.2% Staff+) e pay gap de 15.6%.", 
     "Plano de Ação Executivo": "Auditar remuneração por nível, estabelecer metas afirmativas em contratação e criar programas de mentoria para liderança técnica feminina."},
    
    {"Pilar Estratégico": "Stack Tecnológico", 
     "Diagnóstico dos Dados": "SQL (57.6%) e Python (54.9%) lideram o mercado, acompanhados de ecossistemas em nuvem (AWS/Azure/GCP).", 
     "Plano de Ação Executivo": "Consolidar arquitetura Lakehouse (Databricks/Snowflake/Cloud) e padronizar pipelines de dados com Python/SQL para acelerar o time to market analítico."},
    
    {"Pilar Estratégico": "Investimento em IA", 
     "Diagnóstico dos Dados": "64.3% dos profissionais utilizam IA, mas esbarram em falta de expertise (36.8%) e dados corporativos despreparados (33.1%).", 
     "Plano de Ação Executivo": "Priorizar investimentos em Governança, Qualidade de Dados e letramento corporativo em IA antes de adquirir ferramentas pontuais de alto custo."}
])

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.55, 0.45],
    specs=[[{"type": "xy"}], [{"type": "table"}]],
    vertical_spacing=0.15,
    subplot_titles=('<b>Principais Barreiras para Escala de IA nas Empresas</b>', '<b>Plano de Ação Executivo para a Diretoria</b>')
)

# Gráfico de Barreiras
fig.add_trace(go.Bar(
    x=df_barreiras['pct_barreiras'], y=df_barreiras['barreira'],
    orientation='h', marker_color='#c92a2a',
    text=df_barreiras.apply(lambda r: f"{r['pct_barreiras']}% ({r['ocorrencias']:,} menções)", axis=1),
    textposition='inside'
), row=1, col=1)

# Tabela Executiva
fig.add_trace(go.Table(
    columnorder=[1, 2, 3],
    columnwidth=[20, 38, 42],
    header=dict(
        values=[f"<b>{c}</b>" for c in resumo_diretoria.columns],
        fill_color='#1f77b4', font=dict(color='white', size=13), align='left', height=30
    ),
    cells=dict(
        values=[resumo_diretoria[k] for k in resumo_diretoria.columns],
        fill_color='#f8f9fa', font=dict(size=12), align='left', height=45
    )
), row=2, col=1)

fig.update_layout(
    template='plotly_white', height=750, showlegend=False,
    title={'text': '<b>Diagnóstico de Risco e Roadmap Estratégico de IA</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 45], title='% das Empresas com Barreiras Relatadas', row=1, col=1)
fig.show()
